# Day 2, Segment 3: Multi-Source Context Retrieval & Intelligent Ranking

## Overview

In real-world applications, relevant information lives in **multiple places**: databases (structured), document stores (unstructured), APIs (real-time), web pages, etc.

This segment teaches you to:
1. **Collect context** from multiple sources simultaneously
2. **Rank intelligently** using multi-factor scoring
3. **Select strategically** using priorities and source types
4. **Generate grounded answers** using diverse context

### The Challenge
- Database context: Structured, high-quality, but static
- Document context: Semantic, comprehensive, but slow to update
- API context: Real-time, accurate, but sparse
- Which source is most relevant? How to combine them?

### The Solution: Multi-Factor Ranking
Score each piece of context by:
- **Relevance**: Keyword/semantic match to query
- **Priority**: Source-assigned importance (1-5)
- **Source weight**: Trust in the source type
- Combine: `score = (relevance × 2 + priority) × source_weight`

### Pipeline Architecture
```
Query Input
    ↓
Collect Context (Database + Documents + API)
    ↓
Score Each Piece (Relevance + Priority + Source Weight)
    ↓
Rank by Score (Highest first)
    ↓
Select Top-K Most Valuable Pieces
    ↓
Format for LLM
    ↓
Generate Answer
```

Let's build this step-by-step!

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("✓ Environment configured")

✓ Environment configured


## Step 1: Data Source - Database Context

**What**: Structured data from your database (customers, revenue, metrics, etc.)

**Characteristics**:
- ✅ Highly reliable (source of truth)
- ✅ Real-time current state
- ✅ Well-structured metadata
- ❌ May be sparse for specific queries
- ❌ Privacy-sensitive (be careful!)

**Example data**:
- Customer segmentation info
- Revenue/financial metrics
- User behavior statistics
- Product configuration

**Metadata fields**:
- `source`: "database" (for routing/weighting)
- `priority`: 1-5 scale (manual importance rating)
- `type`: "structured" vs "unstructured"

In [2]:
def get_db_context():
    """
    Retrieve context from structured database.
    
    Returns:
        List of context dicts with content, source, priority, type
    """
    return [
        {
            "content": "Enterprise customers generate 70% of total revenue.",
            "source": "database",
            "priority": 5,  # High priority: mission-critical insight
            "type": "structured"
        }
    ]


# Test: Get database context
db_context = get_db_context()
print("✓ Database context retrieved")
for item in db_context:
    print(f"  - Priority {item['priority']}: {item['content']}")

✓ Database context retrieved
  - Priority 5: Enterprise customers generate 70% of total revenue.


## Step 2: Data Source - Document Context

**What**: Unstructured documents (knowledge base, guides, past decisions, etc.)

**Characteristics**:
- ✅ Comprehensive and detailed
- ✅ Semantic richness (narrative context)
- ✅ Easy to add and update
- ❌ No enforced structure
- ❌ May have redundancy
- ❌ Harder to validate accuracy

**Example data**:
- Product strategy documents
- Technical guides
- Meeting notes
- Case studies
- Pricing rationales

**Metadata fields**:
- `source`: "documents"
- `priority`: Varies by document importance
- `type`: "unstructured"

In [3]:
def get_document_context():
    """
    Retrieve context from unstructured document store.
    
    Returns:
        List of context dicts from knowledge base
    """
    return [
        {
            "content": "Subscription pricing works best for enterprise SaaS.",
            "source": "documents",
            "priority": 3,
            "type": "unstructured"
        },
        {
            "content": "Freemium models often fail in enterprise environments.",
            "source": "documents",
            "priority": 4,
            "type": "unstructured"
        }
    ]


# Test: Get document context
doc_context = get_document_context()
print(f"✓ Retrieved {len(doc_context)} document(s)")
for item in doc_context:
    print(f"  - Priority {item['priority']}: {item['content']}")

✓ Retrieved 2 document(s)
  - Priority 3: Subscription pricing works best for enterprise SaaS.
  - Priority 4: Freemium models often fail in enterprise environments.


## Step 3: Data Source - API Context

**What**: Real-time data from external APIs (market data, competitor info, trends, etc.)

**Characteristics**:
- ✅ Up-to-the-minute information
- ✅ External perspective (competitors, market)
- ✅ Quantifiable metrics
- ❌ May be noisy or unreliable
- ❌ Latency concerns (API calls)
- ❌ Limited historical context

**Example data**:
- Competitor pricing changes
- Market trends
- Customer feedback sentiment
- Usage statistics
- Real-time alerts

**Metadata fields**:
- `source`: "api"
- `priority`: Varies by importance
- `type`: "real-time"

In [4]:
def get_api_context():
    """
    Retrieve context from real-time external APIs.
    
    Returns:
        List of context dicts from live data sources
    """
    return [
        {
            "content": "Competitor recently launched usage-based pricing.",
            "source": "api",
            "priority": 4,
            "type": "real-time"
        }
    ]


# Test: Get API context
api_context = get_api_context()
print(f"✓ Retrieved {len(api_context)} real-time update(s)")
for item in api_context:
    print(f"  - Priority {item['priority']}: {item['content']}")

✓ Retrieved 1 real-time update(s)
  - Priority 4: Competitor recently launched usage-based pricing.


## Step 4: Collect All Context

**Goal**: Aggregate context from all sources into a single list.

**Pattern**: Source-agnostic collection
- Database context
- Document context  
- API context
- (Future: Web context, email, etc.)

**Why it matters**:
- Unified interface for downstream processing
- Easy to add/remove data sources
- Separates collection from ranking

In [5]:
def collect_all_context():
    """
    Aggregate context from all sources.
    
    Returns:
        List of all context pieces from database, documents, and API
    """
    context = []
    context.extend(get_db_context())
    context.extend(get_document_context())
    context.extend(get_api_context())

    return context


# Test: Collect all context
all_context = collect_all_context()
print(f"✓ Collected {len(all_context)} total context piece(s)")
print("\nContext summary by source:")
for source in ["database", "documents", "api"]:
    count = sum(1 for c in all_context if c["source"] == source)
    print(f"  - {source.capitalize()}: {count}")
print(f"\nTotal: {len(all_context)}")

✓ Collected 4 total context piece(s)

Context summary by source:
  - Database: 1
  - Documents: 2
  - Api: 1

Total: 4


## Step 5: Relevance Scoring

**Goal**: Measure how relevant each context piece is to the user's query.

**Simple strategy**: Keyword matching
- Count how many query words appear in context
- Higher count = more relevant

**Why it matters**:
- First factor in final ranking
- Can be replaced with embeddings for better results
- Foundation for multi-factor scoring

**Note**: This is intentionally simple for clarity. In production, use:
- Semantic similarity (embedding-based)
- BM25 ranking
- Neural networks

**Example**:
```
Query: "pricing strategy enterprise"
Context: "Subscription pricing works best for enterprise"
Match: "pricing" (1) + "enterprise" (1) = 2 matches
```

In [6]:
def compute_relevance_score(text, query):
    """
    Compute relevance of text to query using keyword matching.
    
    Args:
        text: Content to score
        query: User query
        
    Returns:
        Relevance score (count of matching words)
    """
    # Simple keyword match (replace with embeddings if needed)
    score = 0

    for word in query.lower().split():
        if word in text.lower():
            score += 1

    return score


# Test: Compute relevance
test_query = "What pricing strategy should we use for enterprise customers?"
test_context = "Subscription pricing works best for enterprise SaaS."

relevance = compute_relevance_score(test_context, test_query)
print(f"✓ Relevance scorer ready")
print(f"  Query: '{test_query}'")
print(f"  Context: '{test_context}'")
print(f"  Relevance score: {relevance} matching words")

✓ Relevance scorer ready
  Query: 'What pricing strategy should we use for enterprise customers?'
  Context: 'Subscription pricing works best for enterprise SaaS.'
  Relevance score: 3 matching words


## Step 6: Multi-Factor Context Ranking (Core Algorithm)

**Goal**: Score each context piece by combining multiple factors.

**Scoring formula**:
```
final_score = (relevance × 2 + priority) × source_weight

Where:
- relevance: Keyword matches (0, 1, 2, ...)
- priority: Source-assigned importance (1-5)
- source_weight: Trust/authority factor
  - API: 1.2 (real-time, high urgency)
  - Database: 1.0 (reliable baseline)
  - Documents: 0.8 (contextual but potentially outdated)
```

**Interpretation**:
- Relevance weighted 2× = Most important factor
- Priority weighted 1× = Medium importance
- Source weight = 20% adjustment based on trust

**Example calculation**:
```
API Context: "Competitor pricing change"
- Relevance: 2 (matches "pricing" + implicit)
- Priority: 4 (high)
- Source weight: 1.2
- Score = (2 × 2 + 4) × 1.2 = 8 × 1.2 = 9.6 ✅ High

Document Context: "Subscription pricing works"
- Relevance: 2 (matches "pricing")
- Priority: 3
- Source weight: 0.8
- Score = (2 × 2 + 3) × 0.8 = 7 × 0.8 = 5.6 ⚠️ Medium
```

**Why this pattern**:
- Multi-factor prevents any single criterion from dominating
- Weights are tunable for different use cases
- Source weighting enables trust-based routing

In [7]:
def rank_context(contexts, query):
    """
    Rank context pieces by multi-factor score: relevance + priority + source weight.
    
    Args:
        contexts: List of context dicts
        query: User query
        
    Returns:
        Contexts sorted by final score (highest first)
    """
    scored = []

    for c in contexts:
        # Factor 1: Relevance (query-content match)
        relevance = compute_relevance_score(c["content"], query)

        # Factor 2: Priority (source-assigned importance)
        base_priority = c.get("priority", 1)

        # Factor 3: Source weighting (trust/authority)
        if c["source"] == "api":
            source_weight = 1.2  # Real-time data: highest urgency
        elif c["source"] == "database":
            source_weight = 1.0  # Reliable baseline
        else:  # documents
            source_weight = 0.8  # Contextual but potentially stale

        # Combine factors: (relevance×2 + priority) × source_weight
        final_score = (relevance * 2 + base_priority) * source_weight

        scored.append((c, final_score))

    # Sort by score (highest first)
    scored.sort(key=lambda x: x[1], reverse=True)

    return [c for c, _ in scored]


# Test: Rank context
test_query = "What pricing strategy should we use for enterprise customers?"
ranked = rank_context(all_context, test_query)

print(f"✓ Ranked {len(ranked)} context piece(s)")
print("\nRanking (highest score first):")
for i, c in enumerate(ranked, 1):
    source = c["source"].upper()
    priority = c["priority"]
    print(f"  {i}. [{source}] Priority {priority}: {c['content'][:50]}...")

✓ Ranked 4 context piece(s)

Ranking (highest score first):
  1. [DOCUMENTS] Priority 3: Subscription pricing works best for enterprise Saa...
  2. [API] Priority 4: Competitor recently launched usage-based pricing....
  3. [DATABASE] Priority 5: Enterprise customers generate 70% of total revenue...
  4. [DOCUMENTS] Priority 4: Freemium models often fail in enterprise environme...


## Step 7: Select Top-K Context

**Goal**: Take only the highest-scoring context pieces to avoid overwhelming the LLM.

**Why top-k selection?**
- Prevents token limit overrun
- Focuses LLM on most relevant info
- Reduces noise and distraction
- Standard in RAG (typically k=3-5)

**Trade-offs**:
- Higher k: More comprehensive context, but slower + higher cost
- Lower k: Faster + cheaper, but may miss important context

**Decision**: k=3 is typical balance

In [8]:
def select_top_context(contexts, k=3):
    """
    Select top-k ranked context pieces.
    
    Args:
        contexts: Ranked context list
        k: Number of pieces to select (default: 3)
        
    Returns:
        Top k context pieces
    """
    return contexts[:k]


# Test: Select top-k
top_k = select_top_context(ranked, k=3)
print(f"✓ Selected top {len(top_k)} context piece(s)")
print("\nSelected context:")
for i, c in enumerate(top_k, 1):
    source = c["source"].upper()
    print(f"  {i}. [{source}] {c['content']}")

✓ Selected top 3 context piece(s)

Selected context:
  1. [DOCUMENTS] Subscription pricing works best for enterprise SaaS.
  2. [API] Competitor recently launched usage-based pricing.
  3. [DATABASE] Enterprise customers generate 70% of total revenue.


## Step 8: Format Context for LLM

**Goal**: Transform structured context into a readable, well-formatted string for the prompt.

**Formatting pattern**:
```
[SOURCE] Content
[SOURCE] Content
[SOURCE] Content
```

**Why source labels matter**:
- LLM understands which context is from which source
- Can weight by source (e.g., "real-time data shows X")
- Improves transparency and traceability
- Helps LLM understand data freshness

In [9]:
def build_context(contexts):
    """
    Format context pieces into a readable string with source labels.
    
    Args:
        contexts: List of selected context pieces
        
    Returns:
        Formatted context string for LLM prompt
    """
    formatted = []

    for c in contexts:
        formatted.append(f"[{c['source'].upper()}] {c['content']}")

    return "\n\n".join(formatted)


# Test: Format context
formatted_context = build_context(top_k)
print(f"✓ Formatted context string")
print(f"  Length: {len(formatted_context)} characters")
print(f"\n--- Formatted Context ---")
print(formatted_context)

✓ Formatted context string
  Length: 185 characters

--- Formatted Context ---
[DOCUMENTS] Subscription pricing works best for enterprise SaaS.

[API] Competitor recently launched usage-based pricing.

[DATABASE] Enterprise customers generate 70% of total revenue.


## Step 9: Generate Answer with LLM

**Goal**: Use the ranked, selected, formatted context to generate a grounded answer.

**Key pattern**:
1. Receive formatted context (multi-source, ranked)
2. Include in system prompt
3. Let LLM synthesize answer

**Why this works**:
- LLM sees the source labels → understands context provenance
- Multi-source context → balanced perspective
- Ranked by relevance → most important info first
- Deterministic (temperature=0) → consistent responses

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def generate_answer(query, context):
    """
    Generate answer using LLM with ranked multi-source context.
    
    Args:
        query: User question
        context: Formatted context string (multi-source, ranked)
        
    Returns:
        Generated answer grounded in context
    """
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0,
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

    messages = [
        SystemMessage(content="You are a senior product strategist."),
        HumanMessage(content=f"""
Use the context below to answer the question. Consider insights from all sources (database, documents, real-time data).

Context:
{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content

/Users/home/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Step 10: Complete Multi-Source Pipeline

**Goal**: Orchestrate all components into a single, reusable pipeline.

**Execution flow**:
```
User Query
    ↓
Collect context from all sources (DB, docs, API)
    ↓
Score by: relevance + priority + source_weight
    ↓
Rank: highest score first
    ↓
Select: top-k pieces only
    ↓
Format: readable string with source labels
    ↓
Generate: LLM answer with context
    ↓
Return: grounded answer to user
```

**Key benefit**: Transparent, modular, auditable
- Can see which sources contributed
- Can tune ranking weights
- Can add new sources easily

In [11]:
def multi_source_pipeline(query):
    """
    Complete pipeline: collect from multiple sources → rank → select → answer.
    
    Args:
        query: User question
        
    Returns:
        Answer grounded in ranked multi-source context
    """
    # Step 1: Collect
    print(f"\n📚 Step 1: Collecting context from all sources...")
    contexts = collect_all_context()
    print(f"   Retrieved {len(contexts)} context piece(s)")

    # Step 2: Rank
    print(f"📊 Step 2: Ranking by relevance + priority + source...")
    ranked = rank_context(contexts, query)
    print(f"   Ranked {len(ranked)} piece(s)")

    # Step 3: Select
    print(f"✂️  Step 3: Selecting top-k most valuable context...")
    top_context = select_top_context(ranked, k=3)
    print(f"   Selected {len(top_context)} piece(s)")

    # Step 4: Transform
    print(f"📝 Step 4: Formatting context for LLM...")
    context_text = build_context(top_context)
    print(f"   Formatted {len(context_text)} character(s)")

    # Step 5: LLM
    print(f"🤖 Step 5: Generating answer with LLM...")
    answer = generate_answer(query, context_text)
    print(f"   Answer generated!")

    return answer

## Step 11: Test the Complete Pipeline

**Scenario**: Product strategist asks about pricing strategy for enterprise.

**What will happen**:
1. Context collected from database (revenue data), documents (pricing guides), and API (competitor moves)
2. Each piece ranked by relevance + priority + source trust
3. Top 3 selected
4. Formatted with source labels
5. Answer generated by LLM

**Expected**: Balanced answer using all three sources

Run the test below:

In [12]:
query = "What pricing strategy should we use for enterprise customers?"

print(f"📝 Query: {query}")

response = multi_source_pipeline(query)

print(f"\n💡 Answer:\n{response}")

📝 Query: What pricing strategy should we use for enterprise customers?

📚 Step 1: Collecting context from all sources...
   Retrieved 4 context piece(s)
📊 Step 2: Ranking by relevance + priority + source...
   Ranked 4 piece(s)
✂️  Step 3: Selecting top-k most valuable context...
   Selected 3 piece(s)
📝 Step 4: Formatting context for LLM...
   Formatted 185 character(s)
🤖 Step 5: Generating answer with LLM...
   Answer generated!

💡 Answer:
Based on the provided context, the most suitable pricing strategy for enterprise customers would be to maintain a subscription-based pricing model. This recommendation is supported by the following insights:

1. **Document Insight**: Subscription pricing is noted to work best for enterprise SaaS, suggesting that this model aligns well with the purchasing preferences and budgetary processes of enterprise clients.

2. **Database Insight**: Enterprise customers are responsible for generating 70% of the total revenue, indicating that they are a critica

---

## Key Takeaways

### 1. **Multi-Source Architecture**
Real-world systems pull from many sources:
- **Database**: Structured truth (financial data, customer info)
- **Documents**: Narrative context (strategy, lessons learned)
- **APIs**: Real-time updates (competitor moves, market data)
- **Web**: External perspectives (news, trends)

### 2. **Multi-Factor Ranking Formula**
```
final_score = (relevance × 2 + priority) × source_weight
```

| Factor | Weight | Purpose |
|--------|--------|---------|
| Relevance | 2× | Query-content match |
| Priority | 1× | Source-assigned importance |
| Source Weight | 1.0-1.2 | Trust/authority/freshness |

### 3. **Source Weighting Strategy**
Different sources have different roles:
- **API (1.2×)**: Real-time is urgent → boost priority
- **Database (1.0×)**: Reliable baseline → neutral weight
- **Documents (0.8×)**: Good context but potentially stale → discount

### 4. **Tunable Parameters**
Adjust these based on your use case:
```python
# Ranking weights
relevance_weight = 2      # Tune if keyword match not ideal
priority_weight = 1       # Tune if priorities not reflecting real importance
source_weights = {...}    # Tune per source type

# Selection
k = 3  # Tune: higher k = broader context, lower k = focused
```

### 5. **Benefits of This Pattern**
- ✅ **Auditable**: See which sources contributed to answer
- ✅ **Tunable**: Adjust weights without code changes (config file)
- ✅ **Scalable**: Easy to add new sources
- ✅ **Balanced**: Prevents any single source from dominating
- ✅ **Real-time**: Latest data can boost relevance immediately

---

## Real-World Scenarios

### Scenario 1: Time-Sensitive Decision
Competitor drops pricing → API context surges in relevance

```python
# Dynamic weighting
if c["source"] == "api" and "competitor" in c["content"]:
    source_weight = 1.5  # Boost urgency
```

### Scenario 2: Regulatory Compliance
Legal documents must always be included

```python
# Enforce minimum inclusion
priority = 5 if "legal" in c.get("type", "") else c["priority"]
```

### Scenario 3: Conflicting Sources
Database says X, but API says Y → LLM synthesizes

```
[DATABASE] Enterprise segment generates 70% revenue
[API] Competitor cutting enterprise pricing by 20%
→ LLM: "While enterprise is our core revenue driver, 
         we must match competitive pricing..."
```

---

## Production Patterns

### Pattern 1: Dynamic Weighting
```python
def get_source_weight(source, recency_score):
    """Weight sources by freshness"""
    base_weights = {"api": 1.2, "database": 1.0, "documents": 0.8}
    
    # Boost if recent
    if recency_score > 0.8:  # Recent (< 1 day)
        return base_weights[source] * 1.1
    return base_weights[source]
```

### Pattern 2: Conflict Resolution
```python
def resolve_conflicts(contexts):
    """If same query is answered differently by sources"""
    grouped = groupby(contexts, key=lambda c: c["topic"])
    
    for topic, group in grouped:
        if len(group) > 1:
            # Multiple answers - prioritize by source weight
            best = max(group, key=lambda c: source_weight[c["source"]])
            include(best)
```

### Pattern 3: Adaptive k Selection
```python
def select_adaptive_k(contexts, max_tokens=500):
    """Select k pieces until approaching token limit"""
    total_tokens = 0
    selected = []
    
    for c in contexts:  # Already ranked
        tokens = count_tokens(c["content"])
        if total_tokens + tokens > max_tokens:
            break
        selected.append(c)
        total_tokens += tokens
    
    return selected
```

---

## Debugging & Monitoring

### Check ranking decisions:
```python
def debug_ranking(contexts, query):
    for c in contexts:
        relevance = compute_relevance_score(c["content"], query)
        priority = c["priority"]
        weight = source_weight[c["source"]]
        score = (relevance * 2 + priority) * weight
        
        print(f"{c['source']:10} | rel={relevance} pri={priority} wt={weight:.1f} → {score:.1f}")
```

### Monitor source contribution:
```python
def source_analysis(top_k):
    sources = [c["source"] for c in top_k]
    print(f"Sources selected: {Counter(sources)}")
    # Should be balanced (not 100% one source)
```

### Audit context quality:
```python
def quality_score(context):
    length = len(context["content"])  # Longer = more detailed
    recency = age_days(context.get("updated_at"))
    specificity = keyword_count(context)  # Domain keywords
    
    return (length/1000 + specificity/10) / (1 + recency/30)
```

---

## Next Steps

In Day 2, Segment 4 (Advanced Retrieval), you'll learn:
- **Query expansion**: Ask multiple reformulations to retrieve more context
- **Metadata filtering**: Reduce context set before ranking
- **Reranking**: Use cross-encoders for fine-grained ranking
- **Hybrid search**: Combine dense (semantic) + sparse (keyword) retrieval

You now have the foundation for production RAG systems! 🚀